# Lab: Coding a Multivariate Linear Regression Workflow

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/okuchap/GB656_2026_public/blob/main/problem-sets/lab-lectures/03-02-lab.ipynb)

This lab prepares you for Problem Set 2 by making the **coding workflow** for multivariate regression predictable. We use hourly Seoul bike rentals as the outcome and a practice feature set that is intentionally different from the final problem-set model. The goal is to learn a workflow that you can transfer, not to copy a finished solution.

## Lab goals

By the end of the lab, you should be able to:

- load a CSV that uses a specified text encoding;
- parse dates and filter rows with a Boolean condition;
- inspect categories and turn text labels into model-ready indicators;
- choose a baseline category and create dummy variables;
- calculate and plot grouped averages;
- combine numeric and dummy-variable features into one design matrix;
- retrieve and interpret multivariate regression results by coefficient name;
- transform future scenarios so their columns match the fitted model; and
- diagnose common encoding, alignment, and interpretation mistakes.

## How to use this notebook

Open the lab using the course Google Colab link and work from top to bottom. Read each explanation before running the code below it. Every code cell has valid starter values, so **Runtime > Restart session and run all** should finish without an error. Exercise prompts ask you to explain or modify a runnable example; restore valid values before continuing.

You do not submit this lab. If you want your changes to persist after you close Colab, select **File > Save a copy in Drive**; otherwise, saving a copy is optional.

Lab 1 already taught the basic pandas inspection tools, Matplotlib scatterplots, the `y`/`X` convention, `sm.add_constant()`, OLS fitting, result attributes, and basic prediction. This lab gives those methods short reminders and concentrates on the new multivariate and categorical-data techniques.


## 0. Setup

Google Colab already includes these packages, so no installation command is needed in a standard runtime. `pandas` handles the table operations, Matplotlib makes the plots, and `statsmodels` fits OLS. `Path` and `quote` support the local-or-public data-loading helper.


In [ ]:
from pathlib import Path
from urllib.parse import quote

import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

plt.style.use("seaborn-v0_8-whitegrid")


## 1. Load and inspect an encoded CSV

`pd.read_csv()` normally assumes UTF-8 text. This file uses a different byte encoding, so a plain read produces a `UnicodeDecodeError`. The argument `encoding="latin1"` tells pandas how to decode the file.

The instructor-provided `course_data_source()` helper first searches the current directory and its parent directories for `data/SeoulBikeData.csv`. If it cannot find a local copy—as in a fresh Colab runtime—it returns the raw-data URL from the public course repository on GitHub. You do not need to upload the CSV or mount Google Drive.


In [ ]:
PUBLIC_REPOSITORY = "okuchap/GB656_2026_public"
PUBLIC_REVISION = "main"


def course_data_source(file_name):
    """Return a local course-data path when available, otherwise its public URL."""
    for root in (Path.cwd(), *Path.cwd().parents):
        local_path = root / "data" / file_name
        if local_path.is_file():
            return local_path

    encoded_name = quote(file_name)
    return (
        "https://raw.githubusercontent.com/"
        f"{PUBLIC_REPOSITORY}/{PUBLIC_REVISION}/data/{encoded_name}"
    )


data_source = course_data_source("SeoulBikeData.csv")
bikes_raw = pd.read_csv(data_source, encoding="latin1")
source_location = (
    "local course repository" if isinstance(data_source, Path) else "public GitHub repository"
)
print(
    f"Loaded {bikes_raw.shape[0]:,} rows and {bikes_raw.shape[1]} columns "
    f"from the {source_location}."
)
bikes_raw.head()


### Check the schema before cleaning

As in Lab 1, check the exact column names and data types before selecting variables. Here the expected raw shape is 8,760 rows by 14 columns. The degree symbols in the two temperature column names should display correctly; that is evidence that the chosen encoding produced readable text.


In [ ]:
assert bikes_raw.shape == (8760, 14)
print(bikes_raw.columns.tolist())
bikes_raw.info()


## 2. Rename the columns

The source names contain spaces, punctuation, and units. Because the schema has already been checked, we can replace the entire column index with a same-length list of concise names. Whole-list assignment is convenient here, but it is safe only when the order and number of source columns are known.

`.copy()` creates the independent working DataFrame `bikes`; the raw table remains available for reference.


In [ ]:
bikes = bikes_raw.copy()

bikes.columns = [
    "date",
    "bike_count",
    "hour",
    "temperature_c",
    "humidity_percent",
    "wind_speed_ms",
    "visibility_10m",
    "dew_point_c",
    "solar_radiation",
    "rainfall_mm",
    "snowfall_cm",
    "season",
    "holiday",
    "functioning_day",
]

assert bikes.columns.is_unique
bikes.head()


**Check:** The first columns should now be `date`, `bike_count`, and `hour`; the last should be `functioning_day`. These exact names are the interface between preprocessing and later model code.


## 3. Parse dates and filter operating hours

The raw `date` values are text in day-month-year order. `pd.to_datetime(..., dayfirst=True)` converts them to pandas datetimes and resolves ambiguous values such as `01/12/2017` as 1 December rather than 12 January.

The system sometimes recorded zero rentals because it was not operating. Those rows describe closure, not ordinary demand, so the analysis keeps only `functioning_day == "Yes"`.

The expression inside `bikes[...]` creates a Boolean mask: one `True` or `False` for each row. Selecting with that mask retains only `True` rows. The familiar `.copy()` then makes the filtered result an independent table that can be modified safely. `pd.api.types.is_datetime64_any_dtype(...)` supplies a focused audit: it returns `True` when the parsed Series has a pandas datetime dtype.


In [ ]:
print("Date dtype before parsing:", bikes["date"].dtype)
print("First raw date:", bikes.loc[0, "date"])
bikes["functioning_day"].value_counts()


In [ ]:
bikes["date"] = pd.to_datetime(bikes["date"], dayfirst=True)
bikes = bikes[bikes["functioning_day"] == "Yes"].copy()

assert pd.api.types.is_datetime64_any_dtype(bikes["date"])
assert (bikes["functioning_day"] == "Yes").all()
assert bikes.shape[0] == 8465

print("Date dtype after parsing:", bikes["date"].dtype)
print(f"Operating-hour observations: {bikes.shape[0]:,}")
bikes[["date", "bike_count", "functioning_day"]].head()


**Check:** There should be 8,465 operating-hour observations. Filtering before creating model features is important: every later Series and dummy-variable table will inherit the same retained row index.


## 4. Inspect and encode categorical features

### 4A. Create a 0/1 holiday indicator

`.value_counts()` reports each distinct label and its frequency. Inspecting labels before encoding catches spelling, capitalization, or unexpected-category problems.


In [ ]:
holiday_counts = bikes["holiday"].value_counts()
assert set(holiday_counts.index) == {"Holiday", "No Holiday"}
holiday_counts


The comparison below is vectorized: it produces `True` for every `Holiday` row and `False` otherwise. `.astype(int)` converts `True` to 1 and `False` to 0, which creates the numeric feature required by the regression.

This mapping makes `No Holiday` the baseline represented by 0. The preceding exact-label check is important: without it, any unexpected text label would also compare unequal to `Holiday` and silently become 0.


In [ ]:
bikes["is_holiday"] = (bikes["holiday"] == "Holiday").astype(int)

assert (bikes.loc[bikes["holiday"] == "Holiday", "is_holiday"] == 1).all()
assert (bikes.loc[bikes["holiday"] == "No Holiday", "is_holiday"] == 0).all()

bikes[["holiday", "is_holiday"]].head()


### Exercise 1 — audit the indicator

A 0/1 indicator has a useful built-in check: its sum equals the number of rows coded 1. Before running the next cell, explain why the two quantities should match. If they do not, the comparison label or filtering step is wrong.


In [ ]:
assert bikes["is_holiday"].sum() == holiday_counts["Holiday"]
print(f"Holiday rows correctly coded as 1: {bikes['is_holiday'].sum():,}")


### 4B. Create season dummy variables with a fixed baseline

`season` has more than two labels. The numeric design-matrix interface `sm.OLS(y, X)` used here cannot accept those strings directly, so we represent the categories with dummy columns.

With an intercept and four categories, use three dummy columns. The omitted category is the **baseline** represented when all three dummies equal 0. We want Winter as that baseline.

`pd.Categorical(..., categories=season_order)` records the full category set and its order. This matters because `pd.get_dummies(..., drop_first=True)` drops the first declared category. Without an explicit order, the omitted category could be determined alphabetically rather than analytically. We also compare the observed labels with the declared set before conversion so an unexpected label cannot silently become a missing category.


In [ ]:
season_counts = bikes["season"].value_counts()
season_counts


In [ ]:
season_order = ["Winter", "Spring", "Summer", "Autumn"]
assert set(season_counts.index) == set(season_order)
bikes["season"] = pd.Categorical(
    bikes["season"],
    categories=season_order,
)

season_dummies = pd.get_dummies(
    bikes["season"],
    prefix="season",
    drop_first=True,
    dtype=int,
)

season_dummies.head()


### Check the baseline and the dummy pattern

The expected columns are `season_Spring`, `season_Summer`, and `season_Autumn`. A Winter row must contain three zeros. Every non-Winter row must contain exactly one 1.

The small codebook below uses the same encoding on one example of each season. `pd.concat([...], axis=1)` places tables side by side and aligns their rows by index; Section 6 uses the same technique to build the model matrix.


In [ ]:
expected_dummy_columns = [
    "season_Spring",
    "season_Summer",
    "season_Autumn",
]

assert season_dummies.columns.tolist() == expected_dummy_columns
assert (
    season_dummies.loc[bikes["season"] == "Winter"].sum(axis=1) == 0
).all()
assert (
    season_dummies.loc[bikes["season"] != "Winter"].sum(axis=1) == 1
).all()

season_codebook = pd.DataFrame(
    {"season": pd.Categorical(season_order, categories=season_order)}
)
season_codebook = pd.concat(
    [
        season_codebook,
        pd.get_dummies(
            season_codebook["season"],
            prefix="season",
            drop_first=True,
            dtype=int,
        ),
    ],
    axis=1,
)
season_codebook


## 5. Summarize and plot the working data

The basic `.describe()` and `.round()` workflow is review from Lab 1. Here it helps verify the outcome, candidate features, and new indicator together. Direct selections below also show how to retrieve the values needed for a short written description of the working sample.

The source records visibility in units of 10 meters. Dividing by 100 creates `visibility_km`, measured in kilometers. This familiar vectorized conversion gives the practice model a readable feature scale without changing the information in the measurements.


In [ ]:
bikes["visibility_km"] = bikes["visibility_10m"] / 100

summary_columns = [
    "bike_count",
    "hour",
    "temperature_c",
    "humidity_percent",
    "wind_speed_ms",
    "visibility_km",
    "solar_radiation",
    "rainfall_mm",
    "snowfall_cm",
    "is_holiday",
]

summary_stats = bikes[summary_columns].describe().round(2)
summary_stats


In [ ]:
n_observations = bikes.shape[0]
n_holiday_categories = holiday_counts.shape[0]
mean_bike_count = bikes["bike_count"].mean()

print(f"Observations: {n_observations:,}")
print(f"Holiday categories: {n_holiday_categories}")
print(f"Average hourly bike count: {mean_bike_count:,.2f}")


### 5A. Review a scatterplot

Matplotlib syntax is review, so the example is brief. We use solar radiation as the practice feature. To transfer the pattern, change the Series supplied on the horizontal axis and update its label and title; the outcome stays on the vertical axis.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

ax.scatter(
    bikes["solar_radiation"],
    bikes["bike_count"],
    alpha=0.25,
)
ax.set_xlabel("Solar radiation (MJ/m²)")
ax.set_ylabel("Hourly rented bikes")
ax.set_title("Bike rentals vs. solar radiation")

plt.show()


### 5B. Calculate and plot grouped means

The next method chain follows a **split–apply–combine** pattern:

1. `.groupby("season", observed=False)` splits rows by season. Because `season` is categorical, `observed=False` retains every declared category even if a particular sample contains no row from one category.
2. `["bike_count"]` selects the outcome to summarize.
3. `.mean()` computes one average per season.
4. `.sort_values()` orders the resulting Series from its smallest mean to its largest.

`Series.plot(kind="bar", ax=ax)` draws those labeled values as bars on a Matplotlib Axes.


In [ ]:
season_means = (
    bikes.groupby("season", observed=False)["bike_count"]
    .mean()
    .sort_values()
)

season_means.round(1)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

season_means.plot(kind="bar", ax=ax)
ax.set_xlabel("Season")
ax.set_ylabel("Average hourly rented bikes")
ax.set_title("Average bike rentals by season")
plt.xticks(rotation=0)

plt.show()


### Exercise 2 — describe, then qualify, the plots

In your notes, write one or two sentences about each plot:

1. What direction or group differences do you see?
2. Is there substantial variation that the plotted feature or category does not explain?
3. Why does neither plot establish that weather or season *causes* a change in rentals?

**Check:** Use association language. These are observational records, and other conditions can differ at the same time.


## 6. Build a multivariable design matrix

This practice model deliberately uses a neighboring feature set rather than the final problem-set specification. It includes time, temperature, visibility in kilometers, solar radiation, the holiday indicator, and the season dummies. You will transfer the same construction pattern to the feature list supplied in the problem set.

`pd.concat([table_a, table_b], axis=1)` combines columns horizontally. Pandas aligns the rows by index labels, so the numeric table and `season_dummies` must come from the same filtered `bikes` DataFrame. If one table were filtered or reset independently, silent missing values or extra rows could appear.

The outcome remains a Series named `y`. The feature table is named `X`, and `sm.add_constant()` adds the intercept column as in Lab 1.


In [ ]:
lab_numeric_features = [
    "hour",
    "temperature_c",
    "visibility_km",
    "solar_radiation",
    "is_holiday",
]

y = bikes["bike_count"]
X = pd.concat(
    [bikes[lab_numeric_features], season_dummies],
    axis=1,
)
X = sm.add_constant(X)

X.head()


### Check the model interface

A valid design matrix has one row per outcome value, a fully numeric set of columns, no missing cells, and an exact known column order. The assertions check the structural requirements, and the displayed dtypes let you confirm that every model column is numeric.


In [ ]:
expected_model_columns = [
    "const",
    "hour",
    "temperature_c",
    "visibility_km",
    "solar_radiation",
    "is_holiday",
    "season_Spring",
    "season_Summer",
    "season_Autumn",
]

assert y.index.equals(X.index)
assert X.columns.tolist() == expected_model_columns
assert X.shape == (8465, 9)
assert X.notna().all().all()
assert (X["const"] == 1).all()

print("Design-matrix checks passed.")
print("X shape:", X.shape)
print("\nColumn dtypes:")
print(X.dtypes)


## 7. Fit OLS and retrieve multivariable results

Model fitting is the same API used in Lab 1: `sm.OLS(y, X)` specifies the model and `.fit()` estimates it. The difference is the wider `X` table. Each coefficient now describes a conditional association—its relationship with predicted rentals **holding the other included features fixed**.


In [ ]:
lab_model = sm.OLS(y, X).fit()
print(lab_model.summary())


### Build a compact coefficient table

The result attributes are review from Lab 1. `params`, `bse`, and `pvalues` are Series indexed by coefficient name; `conf_int()` returns confidence-interval endpoints with the same row labels. Combining them into one DataFrame makes name-based selection convenient.


In [ ]:
conf_int = lab_model.conf_int()

coef_table = pd.DataFrame(
    {
        "estimate": lab_model.params,
        "std_error": lab_model.bse,
        "p_value": lab_model.pvalues,
        "ci_lower": conf_int[0],
        "ci_upper": conf_int[1],
    }
)

selected_terms = [
    "const",
    "temperature_c",
    "solar_radiation",
    "season_Autumn",
]
coef_table.loc[selected_terms]


### Select the quantities a written analysis needs

Select by coefficient label rather than numerical row position. Labels remain correct if the feature order changes. The code below retrieves examples of an intercept, a continuous feature, another weather feature, a dummy coefficient, uncertainty measures, and in-sample fit.

If a p-value displays as `0.0000` after rounding, it is not literally zero; scientific notation such as `.3g` preserves its scale.


In [ ]:
season_term = "season_Autumn"  # Try Spring or Summer after the first run.
weather_term = "solar_radiation"

intercept = lab_model.params["const"]
temperature_coef = lab_model.params["temperature_c"]
temperature_se = lab_model.bse["temperature_c"]
temperature_p_value = lab_model.pvalues["temperature_c"]
temperature_ci = lab_model.conf_int().loc["temperature_c"]
season_coef = lab_model.params[season_term]
weather_coef = lab_model.params[weather_term]
r_squared = lab_model.rsquared

print(f"Intercept: {intercept:,.2f}")
print(f"Temperature coefficient: {temperature_coef:,.2f}")
print(f"Temperature standard error: {temperature_se:,.2f}")
print(f"Temperature p-value: {temperature_p_value:.3g}")
print(
    "Temperature 95% CI: "
    f"[{temperature_ci.iloc[0]:,.2f}, {temperature_ci.iloc[1]:,.2f}]"
)
print(f"{season_term} coefficient: {season_coef:,.2f}")
print(f"{weather_term} coefficient: {weather_coef:,.2f}")
print(f"R-squared: {r_squared:.3f}")


### Exercise 3 — translate output into a careful interpretation

Use the printed values to practice these templates:

- **Intercept:** the prediction when all numeric features equal 0, `is_holiday` is 0, and all season dummies are 0 (Winter). Decide whether that combination is realistic before giving the intercept a business meaning.
- **Continuous coefficient:** a one-unit higher feature value is associated with an estimated coefficient-sized change in predicted hourly rentals, **holding the other included features fixed**. Always state the feature unit.
- **Dummy coefficient:** the named category differs from the omitted Winter baseline by the estimated amount, holding the other features fixed.
- **Standard error:** the estimate's sampling variability across hypothetical repeated samples under the model assumptions.
- **P-value:** evidence against a zero-coefficient null under the model assumptions; it is not the probability that the null hypothesis is true.
- **95% confidence interval:** a range generated by a procedure that would contain the population coefficient in about 95% of repeated samples under the assumptions.
- **$R^2$:** the fraction of outcome variation explained **in the fitted sample**. It does not by itself measure forecasting error on new data.

Write one sentence for `temperature_c`, one for `season_term`, and one for `weather_term`. Use association or prediction language rather than causal language.


## 8. Predict for several practice scenarios

Future data must undergo the same feature transformations used for training. The scenarios below are deliberately different from the required problem-set cases and use the practice model's feature set.


In [ ]:
new_scenarios = pd.DataFrame(
    {
        "scenario": [
            "Cool spring holiday lunch",
            "Bright summer afternoon",
            "Clear autumn evening",
            "Sunny winter noon",
        ],
        "hour": [12, 15, 18, 12],
        "temperature_c": [12, 29, 18, 3],
        "visibility_km": [18, 20, 20, 19],
        "solar_radiation": [1.4, 2.7, 0.1, 1.0],
        "is_holiday": [1, 0, 0, 0],
        "season": ["Spring", "Summer", "Autumn", "Winter"],
    }
)

new_scenarios


### Apply the training transformations in the same order

1. Apply the full `season_order` to the new season column.
2. Call `pd.get_dummies()` with the same prefix, baseline rule, and integer dtype.
3. Concatenate the same numeric features and season dummies.
4. Add the intercept. `has_constant="add"` forces a `const` column even when a small batch makes another feature look constant by accident.
5. Use `.reindex(columns=X.columns, fill_value=0)` to enforce the fitted model's exact column names and order. Any expected dummy absent from a future batch is created and filled with 0; unrelated columns are excluded.

Matching column names is not enough—column order must also match because the fitted coefficients correspond to the ordered columns of `X`.


In [ ]:
new_scenarios["season"] = pd.Categorical(
    new_scenarios["season"],
    categories=season_order,
)

new_season_dummies = pd.get_dummies(
    new_scenarios["season"],
    prefix="season",
    drop_first=True,
    dtype=int,
)

new_X = pd.concat(
    [new_scenarios[lab_numeric_features], new_season_dummies],
    axis=1,
)
new_X = sm.add_constant(new_X, has_constant="add")
new_X = new_X.reindex(columns=X.columns, fill_value=0)

new_X


### Check before calling `predict()`

These assertions verify the prediction interface directly. If one fails, fix the transformation rather than renaming columns by trial and error.


In [ ]:
assert new_X.columns.tolist() == X.columns.tolist()
assert new_X.index.equals(new_scenarios.index)
assert new_X.notna().all().all()
print("Practice-scenario columns match the fitted model.")


In [ ]:
new_scenarios["predicted_bike_count"] = lab_model.predict(new_X)

practice_predictions = new_scenarios[
    ["scenario", "predicted_bike_count"]
].sort_values("predicted_bike_count", ascending=False)

practice_predictions.round(1)


### Exercise 4 — read the prediction table

Identify the highest and lowest practice predictions. Check whether the ranking is plausible from the scenario values, but remember that the fitted model treats hour and every numeric feature as linear and additive. A decimal prediction is valid: it is an estimated average count, not a claim that a fraction of a bike will be rented.


## 9. Build and predict one additional scenario

A one-row DataFrame needs every dictionary value inside a one-element list. Bare scalar values cause pandas to raise a `ValueError` because it cannot infer the row index.

The starter case below is valid and keeps the notebook runnable. Change the scenario name and at least two feature values, then rerun this section. Keep `hour` from 0 through 23, use nonnegative visibility and solar radiation, set `is_holiday` to 0 or 1, and choose a season from `season_order`.


In [ ]:
my_scenario = pd.DataFrame(
    {
        "scenario": ["Mild spring afternoon"],  # Exercise: rename.
        "hour": [14],  # Exercise: change at least two feature values.
        "temperature_c": [17],
        "visibility_km": [19],
        "solar_radiation": [1.2],
        "is_holiday": [0],
        "season": ["Spring"],
    }
)

my_scenario


Apply the same five transformation steps. Using the full category order is especially useful for a one-row table: `get_dummies()` still creates the complete training dummy schema even though only one season is observed.


In [ ]:
my_scenario["season"] = pd.Categorical(
    my_scenario["season"],
    categories=season_order,
)

my_season_dummies = pd.get_dummies(
    my_scenario["season"],
    prefix="season",
    drop_first=True,
    dtype=int,
)

my_X = pd.concat(
    [my_scenario[lab_numeric_features], my_season_dummies],
    axis=1,
)
my_X = sm.add_constant(my_X, has_constant="add")
my_X = my_X.reindex(columns=X.columns, fill_value=0)

assert my_X.columns.tolist() == X.columns.tolist()
print("One-row scenario columns match the fitted model.")
my_X


In [ ]:
my_scenario["predicted_bike_count"] = lab_model.predict(my_X)
my_scenario[["scenario", "predicted_bike_count"]].round(1)


### Exercise 5 — assess your scenario

Describe the values you chose, report the prediction, and decide whether it seems plausible relative to the multi-row practice predictions. A passed shape check proves only that the model *can* calculate the prediction; it does not prove that the scenario is realistic or that the forecast is accurate.


## 10. Transfer the workflow to a business analysis

A concise business summary should connect the model to decisions without overstating it. In four to six sentences, cover:

1. which included features appear useful for predicting system-wide hourly demand;
2. how an operator could use forecasts for staffing, maintenance, or broad bike availability;
3. whether the model is ready to be a final operational forecasting tool; and
4. what data or modeling changes could improve it.

Important limitations include the linear treatment of hour, the absence of station-level location and inventory information, and the lack of out-of-sample evaluation. Coefficients from observational data describe fitted associations, not intervention effects.


## 11. Common mistakes and quick diagnoses

| Symptom | Likely cause | Fix |
|---|---|---|
| Data loading raises an HTTP or file error | Neither a local course copy nor the public GitHub copy is reachable | Reopen the course Colab link, confirm that the runtime has internet access, and rerun the setup and loading cells |
| `UnicodeDecodeError` | The source file was read as UTF-8 | Use `encoding="latin1"` for this CSV |
| Parsed dates swap month and day | The source format was interpreted month-first | Use `pd.to_datetime(..., dayfirst=True)` |
| Closed-system zeros remain in the model | The Boolean filter was not assigned back to the working table | Keep only `functioning_day == "Yes"` before making features |
| A text column reaches `sm.OLS()` | A categorical variable was not encoded | Create a 0/1 indicator or dummy variables with integer dtype |
| Winter has its own dummy or the wrong baseline is omitted | Category order or `drop_first=True` is wrong | Declare `season_order` with Winter first, then recreate the dummies |
| `X` contains missing rows after concatenation | Numeric and dummy tables have different indexes | Create both from the same filtered DataFrame and inspect index equality |
| The model summary has no `const` row | The intercept was not added | Run `X = sm.add_constant(X)` before fitting |
| Prediction reports a shape or column-name error | Future data do not match fitted columns | Repeat the training encoding and reindex to `X.columns` |
| A one-row scenario does not get `const` | Automatic constant detection mistook another column for a constant | Use `has_constant="add"` |
| Interpretation compares a season with the wrong group | The omitted baseline was forgotten | Read dummy coefficients relative to Winter |
| A coefficient is described as causal | Observational association was overstated | Say “is associated with” and “holding the other included features fixed” |
| The notebook works only out of order | Hidden session state remains from earlier experiments | Restart the session and run all cells from top to bottom |


## 12. Final transfer checklist

For the problem set, adapt—not copy—the practice workflow:

1. load the encoded CSV, rename columns, parse dates, and filter operating hours;
2. inspect labels, build the holiday indicator, and create Winter-baseline season dummies;
3. calculate the requested summaries and plots;
4. replace `lab_numeric_features` with the supplied problem-set feature list;
5. concatenate numeric and dummy features, add the intercept, and run the interface checks;
6. fit OLS and select required results by coefficient name;
7. interpret continuous and dummy coefficients while holding the other features fixed;
8. encode, align, and predict the required scenarios;
9. create and assess one realistic one-row scenario; and
10. write the business recommendation and limitations in your own words.

Before submitting any notebook, restart the session, run all cells from top to bottom, and confirm that every plot, model result, check, and prediction appears without an error.


## 13. Tips for Problem Set 2

You've now practiced essentially the same workflow you will use in Problem Set 2. The main differences are the features and prediction scenarios:

| In this lab | In Problem Set 2 |
|---|---|
| use a practice set of numeric features | use the features specified in the problem set |
| create holiday and season indicators | Same |
| build `y` and a multivariable `X` | Same |
| add an intercept with `sm.add_constant()` | Same |
| fit OLS with `sm.OLS(y, X).fit()` | Same |
| interpret coefficients, uncertainty, and $R^2$ | Same |
| build and align prediction DataFrames | Same |
| predict rentals for example scenarios | use the scenarios specified in the problem set |

The main idea you will transfer is using the same multivariate regression workflow with the features and scenarios specified in Problem Set 2.